# 🧾 Scanntech QA Validator

**Validador automatizado de cupons fiscais e promoções Scanntech**

Execute as células **em ordem**. Cada bloco tem instruções claras.

---
| Etapa | O que faz |
|---|---|
| 1️⃣ Dependências | Instala libs e clona o repositório |
| 2️⃣ Upload | Faz upload dos 3 arquivos de entrada |
| 3️⃣ Execução | Roda o validador completo |
| 4️⃣ Download | Baixa o TEMPLATE preenchido + log JSON |


## 1️⃣ Instalação e Clone do Repositório

In [ ]:
# Instala dependências
!pip install openpyxl pandas pdfplumber -q
print('✅ Dependências instaladas')

In [ ]:
import os

# Clona o repositório (ou faz pull se já existir)
if not os.path.exists('automacao_scann'):
    !git clone https://github.com/Miked0/automacao_scann.git
    print('✅ Repositório clonado')
else:
    %cd automacao_scann
    !git pull origin main
    %cd ..
    print('✅ Repositório atualizado')

%cd automacao_scann
print(f'📁 Diretório atual: {os.getcwd()}')

## 2️⃣ Upload dos Arquivos de Entrada

> **Faça upload dos 3 arquivos obrigatórios:**
> - 📋 `TEMPLATE_COM_BIN_NOVO.xlsx` — Roteiro de testes
> - 📊 `export_tickets_audit_companyId-200056_....xlsx` — Export Audit
> - 📄 `ilovepdf_merged-1-2.pdf` — PDF com cupons fiscais
>
> ⚠️ Os nomes dos arquivos serão detectados automaticamente.

In [ ]:
from google.colab import files as colab_files
import shutil, pathlib

print('📂 Selecione os arquivos: TEMPLATE xlsx, Audit xlsx e PDF de cupons')
uploaded = colab_files.upload()

# Move os arquivos para o diretório do projeto
for nome, conteudo in uploaded.items():
    destino = pathlib.Path(nome).name
    with open(destino, 'wb') as f:
        f.write(conteudo)
    print(f'  ✅ {destino} ({len(conteudo)/1024:.1f} KB)')

print('\n📦 Arquivos disponíveis na pasta:')
for f in sorted(pathlib.Path('.').glob('*')):
    if f.suffix in ('.xlsx', '.pdf'):
        print(f'  📄 {f.name}')

In [ ]:
import pathlib

# Detecta automaticamente os arquivos corretos
xlsx_files = sorted(pathlib.Path('.').glob('*.xlsx'))
pdf_files  = sorted(pathlib.Path('.').glob('*.pdf'))

# Separa TEMPLATE (roteiro) do Audit
roteiro_candidates = [f for f in xlsx_files if 'template' in f.name.lower() or 'roteiro' in f.name.lower()]
audit_candidates   = [f for f in xlsx_files if 'audit' in f.name.lower() or 'export' in f.name.lower()]

# Fallback: se não achar pela keyword, pega os dois primeiros xlsx
if not roteiro_candidates and len(xlsx_files) >= 1:
    roteiro_candidates = [xlsx_files[0]]
if not audit_candidates and len(xlsx_files) >= 2:
    audit_candidates = [xlsx_files[1]]

ROTEIRO = str(roteiro_candidates[0]) if roteiro_candidates else None
AUDIT   = str(audit_candidates[0])   if audit_candidates   else None
PDF     = str(pdf_files[0])          if pdf_files          else None
OUTPUT  = 'TEMPLATE_PREENCHIDO.xlsx'
LOG     = 'qa_audit_log.json'

print('🔍 Arquivos detectados:')
print(f'  📋 Roteiro : {ROTEIRO}')
print(f'  📊 Audit   : {AUDIT}')
print(f'  📄 PDF     : {PDF}')
print(f'  💾 Output  : {OUTPUT}')
print(f'  📝 Log     : {LOG}')

if not all([ROTEIRO, AUDIT, PDF]):
    print('\n⚠️  ATENÇÃO: algum arquivo não foi detectado. Verifique os uploads acima.')
else:
    print('\n✅ Tudo pronto para execução!')

## 3️⃣ Execução do Validador

> Essa célula roda o script principal e exibe o log em tempo real.

In [ ]:
# Validação de segurança antes de rodar
assert ROTEIRO and pathlib.Path(ROTEIRO).exists(), f'❌ Roteiro não encontrado: {ROTEIRO}'
assert AUDIT   and pathlib.Path(AUDIT).exists(),   f'❌ Audit não encontrado: {AUDIT}'
assert PDF     and pathlib.Path(PDF).exists(),      f'❌ PDF não encontrado: {PDF}'
print('✅ Todos os arquivos confirmados. Iniciando validação...\n')

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, 'scanntech_qa_validator.py',
    '--roteiro', ROTEIRO,
    '--audit',   AUDIT,
    '--pdf',     PDF,
    '--output',  OUTPUT,
    '--log',     LOG,
]

result = subprocess.run(cmd, capture_output=False, text=True)

if result.returncode == 0:
    print('\n✅ Validação concluída com sucesso!')
else:
    print(f'\n❌ Erro na execução (código {result.returncode}). Verifique o log acima.')

## 4️⃣ Resumo dos Resultados

In [ ]:
import json, pathlib

log_path = pathlib.Path(LOG)
if not log_path.exists():
    print('⚠️  Log JSON não encontrado. A execução pode ter falhado.')
else:
    with open(log_path, encoding='utf-8') as f:
        entries = json.load(f)

    total  = len(entries)
    passou = sum(1 for e in entries if e.get('passou'))
    falhou = total - passou
    pct    = round(passou / total * 100, 1) if total else 0

    print('=' * 50)
    print('📊 RESUMO DA VALIDAÇÃO')
    print('=' * 50)
    print(f'  Total de testes : {total}')
    print(f'  ✅ Passaram      : {passou}  ({pct}%)')
    print(f'  ❌ Falharam      : {falhou}')
    print('=' * 50)

    if falhou > 0:
        print('\n🔴 Testes com falha:')
        for e in entries:
            if not e.get('passou'):
                print(f"  Linha {e['linha']:>3} | Etapa: {e['etapa']} | {e['motivo_erro']}")

## 5️⃣ Download dos Resultados

In [ ]:
from google.colab import files as colab_files
import pathlib

artefatos = [
    (OUTPUT, '📋 TEMPLATE preenchido (xlsx)'),
    (LOG,    '📝 Log de auditoria (JSON)'),
    ('qa_validation.log', '📄 Log textual (.log)'),
]

for caminho, label in artefatos:
    p = pathlib.Path(caminho)
    if p.exists():
        print(f'⬇️  Baixando {label}: {caminho}')
        colab_files.download(caminho)
    else:
        print(f'⚠️  {label} não encontrado: {caminho}')

---

## ⚙️ Execução Avançada (Opcional)

> Use essa célula se quiser passar um diretório de JSONs avulsos com `--json-dir`.

In [ ]:
# Opcional: upload de JSONs avulsos
# Descomente e execute se o parceiro enviar JSONs separados do Audit

# import os
# os.makedirs('jsons_avulsos', exist_ok=True)
# %cd jsons_avulsos
# from google.colab import files as colab_files
# uploaded_json = colab_files.upload()
# for nome, conteudo in uploaded_json.items():
#     with open(nome, 'wb') as f: f.write(conteudo)
# %cd ..

# !python scanntech_qa_validator.py \
#   --roteiro  "{ROTEIRO}" \
#   --audit    "{AUDIT}" \
#   --pdf      "{PDF}" \
#   --output   "{OUTPUT}" \
#   --log      "{LOG}" \
#   --json-dir "jsons_avulsos"

print('Célula avançada disponível — descomente para usar.')